## Подключение и анализ сырых данных

In [13]:
import pandas as pd
from pathlib import Path

In [16]:
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR
DATA_DIR = PROJECT_ROOT/"data"/"CFPB_data"


In [23]:
data_files=sorted(DATA_DIR.glob("complaints-*.csv"))
print("Количество файлов:", len(data_files))
for file in data_files:
    print(file.name)
print("Каждый файл отвечает за данные, собранные за определеннный год и месяц.")

Количество файлов: 6
complaints-2024-01.csv
complaints-2024-02.csv
complaints-2024-03.csv
complaints-2025-01.csv
complaints-2025-02.csv
complaints-2025-03.csv
Каждый файл отвечает за данные, собранные за определеннный год и месяц.


## Ознакомление с полями и типами данных в записях

Все таблицы имеют одинаковый набор полей, поэтому будет достаточно одной таблицы для ознакомления. Возьмем март 2025 года

In [28]:
example=pd.read_csv(
    data_files[5],
    nrows=5
)
example.columns.tolist()
example.dtypes

Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Company public response           str
Company                           str
State                             str
ZIP code                        int64
Tags                              str
Submitted via                     str
Date sent to company              str
Company response to consumer      str
Timely response?                  str
Complaint ID                    int64
dtype: object

#### Видим 2 проблемы:
1. Значения в полях "ZIP code" и "Complaint ID" относятся в числовому типу int64, из-за чего могут возникнуть ошибки, поскольку значения могут начинаться с нуля
2. Значения в поле "Date received" и "Date sent to company" имеют тип str, а не datetime. Это может быть вызвано тем, что csv не хранит исходный тип данных

In [36]:
table=[]
for file in data_files:
    cur=pd.read_csv(
        file,
        dtype={
            "Complaint ID": "string",
            "ZIP code": "string",
        },
        low_memory=False,
    )
    table.append(cur)

## Объединение таблиц и присвоение правильных типов datetime

In [37]:
df = pd.concat(
    table,
    ignore_index=True,
)
df["Date received"] = pd.to_datetime(
    df["Date received"],
    errors="coerce",
)

df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"],
    errors="coerce",
)
print("Количество строк:", df.shape[0])
print("Количество столбцов:", df.shape[1])
df.head()

Количество строк: 141822
Количество столбцов: 15


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-01-03 14:57:29+00:00,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Transaction was not authorized,Company believes it acted appropriately as aut...,NAVY FEDERAL CREDIT UNION,VA,22191,Servicemember,Web,2024-01-03 15:25:33+00:00,Closed with explanation,Yes,8101341
1,2024-01-01 00:24:12+00:00,Checking or savings account,Checking account,Problem caused by your funds being low,Bounced checks or returned payments,Company has responded to the consumer and the ...,FIRST TECHNOLOGY FEDERAL CREDIT UNION,TX,76052,NaN,Web,2024-01-03 21:07:17+00:00,Closed with explanation,Yes,8085678
2,2024-01-01 01:53:02+00:00,Credit card,General-purpose credit card or charge card,Improper use of your report,Reporting company used your report improperly,NaN,CAPITAL ONE FINANCIAL CORPORATION,GA,31537,NaN,Web,2024-01-01 02:02:02+00:00,Closed with explanation,Yes,8085114
3,2024-01-01 03:52:22+00:00,Checking or savings account,Checking account,Opening an account,Account opened without my consent or knowledge,NaN,Chime Financial Inc,CA,95330,NaN,Web,2024-01-01 04:01:19+00:00,Closed with explanation,Yes,8085323
4,2024-01-01 04:04:07+00:00,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,Company has responded to the consumer and the ...,TRUIST FINANCIAL CORPORATION,FL,336XX,NaN,Web,2024-01-01 04:20:58+00:00,Closed with explanation,Yes,8085280


## Разделение полноцнной даты на составные

In [44]:
df["year"] = df["Date received"].dt.year
df["month"] = df["Date received"].dt.month
df["year_month"] = df["Date received"].dt.to_period("M")

/var/folders/3r/djdyxpxx7_57tqk6xx3bf5c80000gn/T/ipykernel_32890/1468834913.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df["Date received"].dt.to_period("M")


#### Проверим типы двнных

In [45]:
df[
    [
        "Date received",
        "year",
        "month",
        "year_month",
    ]
].dtypes

Date received    datetime64[us, UTC]
year                           int32
month                          int32
year_month                 period[M]
dtype: object